In [0]:
# %pip install -U -qqqq databricks-vectorsearch mlflow
# %restart_python

In [ ]:
import sys
import os
from pathlib import Path

def add_project_root_to_sys_path(marker_file: str = ".env"):
    """
    Walk up the directory tree until a folder containing `marker_file` is found.
    Then add that folder to sys.path.
    """
    current = Path.cwd()
    for parent in [current] + list(current.parents):
        if (parent / marker_file).exists():
            root_path = str(parent.resolve())
            if root_path not in sys.path:
                sys.path.insert(0, root_path)
                print(f"Added to sys.path: {root_path}")
            return
    print(f"Project root with {marker_file} not found.")

add_project_root_to_sys_path()

In [0]:
from databricks.vector_search.client import VectorSearchClient
from datetime import timedelta
import time

In [0]:
from configs.project import get_project_config
# from src.data_utils import get_df_from_config


projectConfig = get_project_config()

In [0]:
vs_config = projectConfig.vector_search_attributes["id_1"]

for k, v in vs_config.model_dump().items():
  print(k, v)

In [0]:
vsc = VectorSearchClient(disable_notice=True)

To create a managed vector search index the source table need to have change data feed enabled. 

In [0]:
spark.sql(f"ALTER TABLE {vs_config.source_table_name} ALTER COLUMN {vs_config.primary_key} SET NOT NULL")
try:
  spark.sql(f"ALTER TABLE {vs_config.source_table_name} ADD CONSTRAINT {vs_config.primary_key}_pk PRIMARY KEY( {vs_config.primary_key} )")
except Exception as e:
  print(f"Constraint {vs_config.primary_key}_pk already exists.")
spark.sql(f"ALTER TABLE {vs_config.source_table_name} SET TBLPROPERTIES (delta.enableChangeDataFeed = true) ")


In [0]:
display(spark.table(vs_config.source_table_name))

In [0]:
try:
    vsc.create_endpoint(name=vs_config.endpoint_name,
                        endpoint_type="STANDARD")
    
    time.sleep(5)

    vsc.wait_for_endpoint(name=vs_config.endpoint_name,
                                timeout=timedelta(minutes=60),
                                verbose=True)
    
    print(f"Endpoint named {vs_config.endpoint_name} is ready.")

    ep = vsc.get_endpoint(name=vs_config.endpoint_name)

except Exception as e:
    if "already exists" in str(e):
        print(f"Endpoint named {vs_config.endpoint_name} already exists.")
        ep = vsc.get_endpoint(name=vs_config.endpoint_name)
    else:
        raise e


In [0]:
from databricks.sdk.service import iam
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
w.permissions.set(request_object_type="vector-search-endpoints",
                  request_object_id=ep["id"],
                  access_control_list=[
                        iam.AccessControlRequest(group_name="users",
                                                   permission_level=iam.PermissionLevel.CAN_MANAGE)
                      ])

Test Embedding Endpoint

In [0]:
import mlflow
import mlflow.deployments

client = mlflow.deployments.get_deploy_client("databricks")

In [0]:
[ep for ep in client.list_endpoints() if ep["name"]==vs_config.embedding_model_endpoint_name]

In [0]:
client.predict(endpoint=vs_config.embedding_model_endpoint_name, inputs={"input": ["What is Apache Spark?"]})

#Create Vector Search Index

https://docs.databricks.com/aws/en/generative-ai/create-query-vector-search#create-index-using-the-python-sdk

https://api-docs.databricks.com/python/vector-search/databricks.vector_search.html#databricks.vector_search.client.VectorSearchClient.create_delta_sync_index_and_wait

In [0]:

try:
  vector_search_index = vsc.create_delta_sync_index_and_wait(
    endpoint_name=vs_config.endpoint_name,
    index_name=vs_config.index_name,
    source_table_name=vs_config.source_table_name,
    primary_key=vs_config.primary_key,
    embedding_source_column=vs_config.embedding_source_column,
    embedding_model_endpoint_name=vs_config.embedding_model_endpoint_name,
    pipeline_type=vs_config.pipeline_type,
    verbose=True
  )
except Exception as e:
    if "already exists" in str(e):
        print(f"Index named {vs_config.endpoint_name} already exists.")
        vector_search_index = vsc.get_index(vs_config.endpoint_name, vs_config.index_name)
    else:
        raise e

In [0]:
spark.sql(f"GRANT SELECT ON TABLE {vs_config.index_name} TO `account users` ")
